# HumMobCov — Main Analysis Notebook

Structured in two sections:
1. **INPUT** — choose the region, load the dataset, inspect configuration
2. **MAIN** — run the full processing pipeline (per-user metric computation)

For visualisation, open `main_visualization_single_user.ipynb`.

All paths, parameters and flags are controlled through:
- `src/constants.py` — global constants and directory layout
- `data/config/config_<REGION>.json` — feature flags (which metrics to compute)


---
## 0 · Imports

In [1]:
import gc
import sys
from pathlib import Path

# Make the src package importable regardless of working directory
PROJECT_ROOT = Path().resolve().parent   # HumMobCov/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Central import hub — brings in numpy, pandas, skmob, matplotlib, etc.
# and all project classes / constants
from src import (
    # constants
    PROJECT_ROOT, DIR_SRC, DIR_OUTPUT, DIR_DATA, DIR_CONFIG,
    PERIOD_NAMES, PERIOD_DIVISION, PERIOD_NAMES_TO_DIVISION,
    MIN_POINTS_PER_USER, TIME_THRESHOLD_HOURS, US_BOUNDING_BOX,
    RURALITY_LEVELS, PARTY_NAMES, K_RADIUS_VALUES, LIST_REGIONS,
    # dataset classes
    DataSet_California, DataSet_Massachusets, dataset_info,
    # pipeline
    analyze_from_dataset, analyze_from_s3_progressive, compute_all, get_config,
    # user
    User,
    # plotter
    plotter,
    # storage
    ParquetStore,
    # utilities
    get_already_saved_user_per_period, ifnotexistsmkdir,
)
from src.constants import (
    DIR_MILESTONES_SERVER,
    S3_ENDPOINT_URL, S3_BUCKET, S3_RAW_PREFIX, DIR_SHARD_TEMP,
    S3_OUTPUT_BUCKET, S3_OUTPUT_PREFIX,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Project root:', PROJECT_ROOT)
print('Output dir:  ', DIR_OUTPUT)
print('Data dir:    ', DIR_DATA)
print('S3 endpoint: ', S3_ENDPOINT_URL)
print('S3 bucket:   ', S3_BUCKET)
print('Out bucket:  ', S3_OUTPUT_BUCKET)


Project root: /home/aamaduzzi/HumMobCov
Output dir:   /home/aamaduzzi/HumMobCov/output
Data dir:     /home/aamaduzzi/HumMobCov/data
S3 endpoint:  https://s3.atlas.fbk.eu
S3 bucket:    chub-datalake
Out bucket:   chub-datalake


---
## 1 · INPUT

**Edit this section** to choose which region to analyse and to review / override any parameter.

In [2]:
# ─── Choose region ──────────────────────────────────────────────────────────
# Options: "CA"  (California)  or  "MA"  (Massachusetts)
REGION = "CA"

assert REGION in LIST_REGIONS, f"Unknown region '{REGION}'. Choose from {LIST_REGIONS}"

In [ ]:
# ─── Optional parameter overrides ───────────────────────────────────────────
# Leave as None to use the defaults from constants.py

OVERRIDE_NP_            = None   # e.g. 30  (minimum stops per user per period)
OVERRIDE_T_THRESHOLD    = None   # e.g. 2   (minimum hours between stops)
OVERRIDE_OUTPUT_DIR     = None   # raw data output — None keeps data in milestones_analysis/{REGION}/
OVERRIDE_CONFIG_DIR     = None   # e.g. Path('/my/configs')

# Set True to skip all computation and go straight to the visualisation cells.
# The parquet store is still initialised so all plot cells work normally.
SKIP_PIPELINE = False

# Plots are saved here (separate from raw data output)
PLOT_DIR = PROJECT_ROOT / "output" / REGION   # → HumMobCov/output/{REGION}/plots/

In [4]:
# ─── Initialise dataset ─────────────────────────────────────────────────────
if REGION == "CA":
    dataset = DataSet_California()
elif REGION == "MA":
    dataset = DataSet_Massachusets()

# Apply overrides
if OVERRIDE_NP_ is not None:
    dataset.np_ = OVERRIDE_NP_
if OVERRIDE_T_THRESHOLD is not None:
    dataset.t_threshold = OVERRIDE_T_THRESHOLD

print(f"Region:              {dataset.id_}")
print(f"Min points (np_):    {dataset.np_}")
print(f"Time threshold (h):  {dataset.t_threshold}")
print(f"Output directory:    {dataset.dir_output}")
print(f"Number of raw files: {len(dataset.dir_files)}")

Region:              CA
Min points (np_):    20
Time threshold (h):  1
Output directory:    /home/aamaduzzi/HumMobCov/milestones_analysis/CA/dataxuser
Number of raw files: 0


In [5]:
# ─── Preview algorithm-flow configuration ───────────────────────────────────
import json

cfg = get_config(REGION, config_dir=OVERRIDE_CONFIG_DIR)
print("Active feature flags:")
for key, val in cfg.items():
    if not key.startswith('_'):
        flag = '✓' if val else '✗'
        print(f"  {flag}  {key}")

Active feature flags:
  ✓  raw_trajectories
  ✓  is_radius_gyration
  ✗  already_computed_rg
  ✓  is_weekly_radius_gyration
  ✓  is_gonzalez
  ✗  already_computed_gonzalez
  ✓  is_random_entropy
  ✗  already_computed_random_entropy
  ✓  is_uncorrelated_entropy
  ✗  already_computed_uncorrelated_entropy
  ✓  is_real_entropy
  ✗  already_computed_real_entropy
  ✓  is_distance
  ✗  already_computed_distance
  ✓  is_home
  ✗  already_computed_home
  ✓  is_krg
  ✗  already_computed_krg
  ✓  is_St
  ✗  already_computed_St
  ✓  is_fraction_time
  ✗  already_computed_fraction_time
  ✓  is_county_rural
  ✗  already_computed_county_rural
  ✓  is_frequency
  ✗  already_computed_frequency
  ✗  is_week2points
  ✓  save_results


In [6]:
# ─── Preview time periods ────────────────────────────────────────────────────
print("Analysis periods:")
for name, (start, end) in PERIOD_NAMES_TO_DIVISION.items():
    print(f"  {name:25s}  {start.date()}  →  {end.date()}")

Analysis periods:
  15 jan - 15 march          2020-01-15  →  2020-03-15
  15 march - 15 may          2020-03-15  →  2020-05-15
  15 may - sept              2020-05-15  →  2020-09-30


---
## 2 · MAIN

Runs the full per-user computation pipeline, selecting the appropriate execution mode automatically:

| Mode | Condition | Action |
|---|---|---|
| **A — local raw** | Raw parquet shards present on local filesystem | `analyze_from_dataset()` — standard batch processing |
| **B — S3 progressive** | `raw_trajectories=true` in config but files not local | `analyze_from_s3_progressive()` — download one shard at a time, compute, delete |
| **C-CA — legacy migration** | CA only; per-user CSV.gz files in `dataxuser/` | `store.migrate_all_periods()` — one-time migration to parquet |
| **C-MA — legacy migration** | MA only; per-metric shard dirs in `milestones_analysis/MA/` | `store.migrate_all_periods_MA()` — one-time migration to parquet |
| **Done** | All periods already in parquet store | Skip — go to Section 3 |

**The pipeline is resume-safe at two levels:**
- *Shard level* (S3 mode): `shard_checkpoint_np_{np_}_t_{t}.json` records which raw files have been fully processed. Interrupted runs restart from the next unprocessed shard.
- *User level*: users already present in the parquet store are detected in O(1) via footer metadata and skipped unconditionally.

**S3 configuration** (override via environment variables if needed):
```
S3_ENDPOINT_URL  https://s3.atlas.fbk.eu   (env: S3_ENDPOINT_URL)
S3_BUCKET        chub-datalake              (env: S3_BUCKET)
S3_PREFIX_CA     shared/cuebiq/MOBS/urban_rural_flow_stops_cali_urban_rural_v3  (env: S3_PREFIX_CA)
S3_PREFIX_MA     shared/cuebiq/MOBS/20220330_stops_hq_users_MA                  (env: S3_PREFIX_MA)
SHARD_TEMP_DIR   HumMobCov/.shard_tmp       (env: SHARD_TEMP_DIR)
```


In [7]:

# ─── Compute-path settings ───────────────────────────────────────────────────
# USE_VECTORIZED=True  → polars/numpy batch path (10-100x faster for most metrics).
#   - Scalar metrics (RG, entropy, distance, home, k-RG, frequency): fully
#     parallel via polars' internal thread pool (uses ALL CPU cores automatically).
#   - Python-callback metrics (real_entropy, S(t), Gonzalez): run serially;
#     parallelization was benchmarked and shown to be slower due to IPC overhead.
# USE_VECTORIZED=False → legacy skmob per-user loop (slower, kept for reference).
USE_VECTORIZED = True

# ─── Initialise parquet store ────────────────────────────────────────────────
store = ParquetStore(
    base_dir     = DIR_MILESTONES_SERVER / REGION,
    np_          = dataset.np_,
    t_threshold  = dataset.t_threshold,
)

# ─── Check what is already computed ──────────────────────────────────────────
# IMPORTANT: completion semantics depend on the execution mode.
#
# raw_trajectories=true (S3 / local raw modes):
#   The unit of work is a raw SHARD.  Completion = every shard in the S3
#   prefix appears in shard_checkpoint_np_{np_}_t_{t}.json.
#   We CANNOT infer this from store contents alone — a partial run (1-of-N
#   shards) produces a fully-consistent store that would look "done".
#   → Always enter the pipeline; analyze_from_s3_progressive /
#     analyze_from_dataset manage their own shard-level resume internally.
#
# raw_trajectories=false (legacy migration):
#   The unit of work is a USER.  Resume is user-granular: already-migrated
#   users are skipped inside migrate_all_periods / migrate_all_periods_MA.
#   → A period is "done" only if all_scalars has any users (safe proxy,
#     because migration never runs unless a period folder is missing).

use_shard_resume = bool(cfg.get("raw_trajectories")) and not SKIP_PIPELINE

if use_shard_resume:
    # Always enter the pipeline; inner shard-level resume handles skipping.
    periods_done = []
    periods_todo = list(PERIOD_NAMES)
else:
    periods_done = [
        p for p in PERIOD_NAMES
        if len(store.get_computed_users(p, "all_scalars")) > 0
    ]
    periods_todo = [p for p in PERIOD_NAMES if p not in periods_done]

print(f"Periods already in store  : {periods_done or 'none'}")
print(f"Periods still to compute  : {periods_todo or 'none'}")
print(f"Compute path              : {'vectorized (polars)' if USE_VECTORIZED else 'legacy (skmob)'}")
print(f"Output bucket             : {S3_OUTPUT_BUCKET}")
print(f"Output prefix             : {S3_OUTPUT_PREFIX[REGION]}")

if SKIP_PIPELINE:
    print("\nSKIP_PIPELINE=True — skipping computation, using existing store data.")
elif not periods_todo:
    print("\nParquet store already populated for all periods. Nothing to do.")

else:
    # ── CASE A: local raw parquet files ──────────────────────────────────────
    local_raw_files = [f for f in dataset.dir_files if Path(f).exists()]

    if cfg.get("raw_trajectories") and local_raw_files:
        print(f"\nMODE A — local raw data ({len(local_raw_files)} shards found).")
        analyze_from_dataset(
            dataset,
            region         = REGION,
            config_dir     = OVERRIDE_CONFIG_DIR,
            output_dir     = OVERRIDE_OUTPUT_DIR,
            store          = store,
            batch_size     = 20000,
            use_vectorized = USE_VECTORIZED,
        )
        print("Computation complete. Consolidating shards…")
        for p in dataset.period_names:
            store.consolidate_all(p)
        print("Uploading results to S3…")
        store.upload_all_to_s3(
            period_names      = dataset.period_names,
            s3_bucket         = S3_OUTPUT_BUCKET,
            s3_prefix         = S3_OUTPUT_PREFIX[REGION],
            endpoint_url      = S3_ENDPOINT_URL,
            delete_after      = False,
            consolidate_first = False,
        )
        print("Done.")

    # ── CASE B: S3 progressive (download one shard → compute → upload → delete) ──
    elif cfg.get("raw_trajectories"):
        print(f"\nMODE B — S3 progressive download.")
        print(f"  Input endpoint  : {S3_ENDPOINT_URL}")
        print(f"  Input bucket    : {S3_BUCKET}")
        print(f"  Input prefix    : {S3_RAW_PREFIX[REGION]}")
        print(f"  Temp dir        : {DIR_SHARD_TEMP}")
        print(f"  Output bucket   : {S3_OUTPUT_BUCKET}")
        print(f"  Output prefix   : {S3_OUTPUT_PREFIX[REGION]}")
        print(f"  Periods         : {periods_todo}\n")
        analyze_from_s3_progressive(
            dataset,
            region                    = REGION,
            cfg                       = cfg,
            store                     = store,
            endpoint_url              = S3_ENDPOINT_URL,
            bucket                    = S3_BUCKET,
            s3_prefix                 = S3_RAW_PREFIX[REGION],
            temp_dir                  = DIR_SHARD_TEMP,
            batch_size                = 20000,
            use_vectorized            = USE_VECTORIZED,
            output_endpoint_url       = S3_ENDPOINT_URL,
            output_bucket             = S3_OUTPUT_BUCKET,
            output_s3_prefix          = S3_OUTPUT_PREFIX[REGION],
            delete_local_after_upload = True,
        )

    # ── CASE C-CA: migrate legacy dataxuser per-user CSV.gz files ────────────
    elif REGION == "CA":
        legacy_dir = DIR_MILESTONES_SERVER / REGION / "dataxuser"
        if legacy_dir.exists() and any(legacy_dir.iterdir()):
            print(
                f"\nMODE C-CA — migrating legacy dataxuser/ CSV.gz files.\n"
                f"  Periods missing: {periods_todo}\n"
                f"  (Already-migrated users are skipped — safe to re-run.)"
            )
            store.migrate_all_periods(
                legacy_dir   = legacy_dir,
                period_names = periods_todo,
                np_          = dataset.np_,
                t            = dataset.t_threshold,
                batch_size   = 20000,
                consolidate  = True,
            )
            print("Uploading results to S3…")
            store.upload_all_to_s3(
                period_names      = dataset.period_names,
                s3_bucket         = S3_OUTPUT_BUCKET,
                s3_prefix         = S3_OUTPUT_PREFIX[REGION],
                endpoint_url      = S3_ENDPOINT_URL,
                delete_after      = False,
                consolidate_first = False,
            )
        else:
            print(
                f"\nCA: no dataxuser/ directory found at {legacy_dir}.\n"
                f"Set raw_trajectories=true in config_CA.json to compute from S3."
            )

    # ── CASE C-MA: migrate legacy per-metric shard directories ───────────────
    elif REGION == "MA":
        ma_legacy_base = DIR_MILESTONES_SERVER / "MA"
        print(
            f"\nMODE C-MA — migrating legacy metric-folder shard files.\n"
            f"  Periods missing: {periods_todo}\n"
            f"  np_={dataset.np_}, t={dataset.t_threshold}\n"
            f"  (Already-migrated users are skipped — safe to re-run.)"
        )
        store.migrate_all_periods_MA(
            ma_legacy_base = ma_legacy_base,
            period_names   = periods_todo,
            np_            = dataset.np_,
            t              = dataset.t_threshold,
            batch_size     = 20000,
            consolidate    = True,
        )
        print("Uploading results to S3…")
        store.upload_all_to_s3(
            period_names      = dataset.period_names,
            s3_bucket         = S3_OUTPUT_BUCKET,
            s3_prefix         = S3_OUTPUT_PREFIX[REGION],
            endpoint_url      = S3_ENDPOINT_URL,
            delete_after      = False,
            consolidate_first = False,
        )

    else:

        print(f"Unknown region {REGION!r}. No action taken.")

Periods already in store  : ['15 jan - 15 march', '15 march - 15 may']
Periods still to compute  : ['15 may - sept']
Compute path              : vectorized (polars)
Output bucket             : chub-datalake
Output prefix             : final_pipeline/CA

SKIP_PIPELINE=True — skipping computation, using existing store data.


In [8]:

# ─── Pipeline summary ────────────────────────────────────────────────────────
# Show how many users are available per period and metric kind.
# When running in S3-progressive mode (use_shard_resume=True) the local
# parquet store is empty because files are deleted after upload.
# In that case we fall back to reading counts directly from S3.

def _count(period, kind, long=False):
    """Return user count from local store; fall back to S3 if local is empty."""
    if long:
        local = store.get_computed_users_long(period, kind)
    else:
        local = store.get_computed_users(period, kind)
    if local:
        return len(local)
    if use_shard_resume:
        s3_users = store.get_s3_computed_users(
            period, kind,
            s3_bucket    = S3_OUTPUT_BUCKET,
            s3_prefix    = S3_OUTPUT_PREFIX[REGION],
            endpoint_url = S3_ENDPOINT_URL,
        )
        return len(s3_users)
    return 0

print("Users available (local store, or S3 if local is empty):")
for period in PERIOD_NAMES:
    scalars  = _count(period, "all_scalars")
    gonzalez = _count(period, "gonzalez",  long=True)
    st       = _count(period, "S")
    wrg      = _count(period, "weekly_rg")
    freq     = _count(period, "frequency", long=True)
    print(
        f"  {period:25s}  "
        f"scalars={scalars:>7,}  "
        f"gonzalez={gonzalez:>7,}  "
        f"S(t)={st:>7,}  "
        f"weekly_rg={wrg:>7,}  "
        f"freq={freq:>7,}"
    )


Users available (local store, or S3 if local is empty):
  15 jan - 15 march          scalars= 18,608  gonzalez= 18,608  S(t)= 18,608  weekly_rg= 18,605  freq= 18,608
  15 march - 15 may          scalars= 14,809  gonzalez= 14,809  S(t)= 14,809  weekly_rg=      0  freq= 14,809
  15 may - sept              scalars=      0  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0


---
## 3 · MIGRATE EXISTING OUTPUT TO S3 (final_pipeline)

**When using MODE B (S3 progressive)**: the pipeline automatically uploads each
shard's output to S3 and deletes local files.  At the very end it also runs a
final consolidation step that merges all per-shard S3 files into a single
`consolidated.parquet` per metric kind per period.

**This section is only needed if:**
- The pipeline ended before the final consolidation ran (e.g. killed after the
  last shard but before the merge step), **or**
- You used MODE A or MODE C (local computation) and the automatic S3 upload in
  Section 2 was skipped or failed.

The canonical output root is:
```
chub-datalake / final_pipeline/{REGION}/
```

Run cells 4.1 and 4.2 as needed.  Already-uploaded files are not re-uploaded.


In [ ]:
# ─── 3.1  Finalise S3 consolidation (MODE B only) ───────────────────────────
# Call this if the pipeline exited after the last shard but before the final
# merge.  For each period it downloads all per-shard S3 files, merges them
# into consolidated.parquet, uploads to S3, and deletes the per-shard files.

UPLOAD_ENDPOINT = S3_ENDPOINT_URL
UPLOAD_BUCKET   = S3_OUTPUT_BUCKET
UPLOAD_PREFIX   = S3_OUTPUT_PREFIX[REGION]

print(f"S3 target : s3://{UPLOAD_BUCKET}/{UPLOAD_PREFIX}")
print(f"Endpoint  : {UPLOAD_ENDPOINT}")
print(f"Region    : {REGION}")

print("\nRunning final S3 consolidation for all periods …")
for period in PERIOD_NAMES:
    print(f"\n  Period: {period!r}")
    store.consolidate_s3_shards(
        period,
        s3_bucket           = UPLOAD_BUCKET,
        s3_prefix           = UPLOAD_PREFIX,
        endpoint_url        = UPLOAD_ENDPOINT,
        delete_shards_after = True,
        verbose             = True,
    )
print("\nFinal consolidation complete.")


In [ ]:
# ─── 3.2  Upload local output to S3 (MODE A / MODE C) ───────────────────────
# If you ran MODE A (local raw files) or MODE C (legacy migration), results are
# in the local ParquetStore.  Use this cell to push them to S3.

periods_on_s3 = store.list_s3_computed_periods(
    s3_bucket    = UPLOAD_BUCKET,
    s3_prefix    = UPLOAD_PREFIX,
    endpoint_url = UPLOAD_ENDPOINT,
)
periods_to_upload = [p for p in PERIOD_NAMES if p not in periods_on_s3]

print(f"Periods already on S3 : {periods_on_s3 or 'none'}")
print(f"Periods to upload     : {periods_to_upload or 'none'}")


In [ ]:
# ── Upload to S3 (MODE A / MODE C) ───────────────────────────────────────────
if not periods_to_upload:
    print("All periods already on S3. Nothing to upload.")
else:
    upload_results = store.upload_all_to_s3(
        period_names      = periods_to_upload,
        s3_bucket         = UPLOAD_BUCKET,
        s3_prefix         = UPLOAD_PREFIX,
        endpoint_url      = UPLOAD_ENDPOINT,
        delete_after      = True,
        consolidate_first = True,
        verbose           = True,
    )
    print("\nUpload summary:")
    for period, kinds in upload_results.items():
        for kind, ok in kinds.items():
            status = "✓" if ok else "✗"
            print(f"  {status}  {period:25s}  {kind}")
